In [ ]:

import numpy as np
from torch.utils.data import BatchSampler, DataLoader, Dataset

In [ ]:
class DummyPatchSpecs:
    data_index: int

In [ ]:
class DummyImageStack:

    def __init__(self, file_id: int):
        self.file_id = file_id
        self.is_loaded: bool = False

    def extract_patch(self):
        if not self.is_loaded:
            self.load()
        patch = f"Patch from image stack {self.file_id}"
        return patch

    def load(self):
        self.is_loaded = True
        print(f"--- Loaded file: {self.file_id} ---")

    def close(self):
        # TODO: raise error if not loaded?
        self.is_loaded = False
        self.extract_patch_calls = 0
        print(f"--- Closed file: {self.file_id} ---")

In [ ]:
class DummyPatchingStrategy:

    def __init__(self, image_stacks: list[DummyImageStack]):
        self.n_patches_per_image_stack = [
            np.random.randint(64, 128) for _ in image_stacks
        ]
        self.patch_bins = np.concatenate(
            [[0], np.cumsum(self.n_patches_per_image_stack)]
        )

    def __getitem__(self, index: int) -> DummyPatchSpecs:
        # minus 1, because digitize returns 1 for first bin
        data_index = np.digitize(index, self.patch_bins) - 1
        return {"data_index": data_index}

In [ ]:
class DummyDataset(Dataset):

    def __init__(self, image_stacks: list[DummyImageStack]):
        self.image_stacks = image_stacks
        self.patching_strategy = DummyPatchingStrategy(image_stacks)

    def __len__(self):
        return sum(self.patching_strategy.n_patches_per_image_stack)

    def __getitem__(self, index):
        patch_specs = self.patching_strategy[index]
        data_index = patch_specs["data_index"]
        patch = self.image_stacks[data_index].extract_patch()
        return patch

In [ ]:
class JitBatchSampler(BatchSampler):

    def __init__(self, dataset: DummyDataset, batch_size: int, n_files_loaded: int = 1):
        self.dataset = dataset
        self.batch_size = batch_size
        self.n_files_loaded = n_files_loaded

    def __iter__(self):
        n_files = len(self.dataset.image_stacks)
        file_indices = np.arange(n_files)
        np.random.shuffle(file_indices)
        file_groups = [
            file_indices[i : i + self.n_files_loaded]
            for i in range(0, n_files, self.n_files_loaded)
        ]
        remaining_indices = None
        for file_group in file_groups:
            patch_indices = np.zeros((0,))
            for file_index in file_group:
                file_patch_indices = np.arange(
                    self.dataset.patching_strategy.patch_bins[file_index],
                    self.dataset.patching_strategy.patch_bins[file_index + 1],
                )
                patch_indices = np.concatenate([patch_indices, file_patch_indices])
            np.random.shuffle(patch_indices)

            if remaining_indices is not None:
                patch_indices = np.concatenate([remaining_indices, patch_indices])
            remaining_indices = None

            for i in range(0, len(patch_indices), self.batch_size):
                batch_indices = list(patch_indices[i : i + self.batch_size])
                if len(batch_indices) == self.batch_size:
                    yield batch_indices
                else:
                    remaining_indices = batch_indices

            patch_bins = self.dataset.patching_strategy.patch_bins
            current_file_indices = np.unique(np.digitize(patch_indices, patch_bins)) - 1
            if remaining_indices is not None:
                remaining_file_indices = (
                    np.unique(np.digitize(remaining_indices, patch_bins)) - 1
                )
            else:
                remaining_file_indices = []
            for file_index in current_file_indices:
                if file_index not in remaining_file_indices:
                    self.dataset.image_stacks[file_index].close()

In [ ]:
image_stacks = [DummyImageStack(i) for i in range(10)]
dataset = DummyDataset(image_stacks)

In [ ]:
dataloader = DataLoader(
    dataset=dataset,
    batch_sampler=JitBatchSampler(dataset=dataset, batch_size=64, n_files_loaded=1),
)

In [ ]:
track_files_open = []
track_batch_size = []
files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
print("--- files open: ", files_open)
for patch_batch in dataloader:
    files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
    track_files_open.append(files_open)
    track_batch_size.append(len(patch_batch))
    print("Batch size: ", len(patch_batch), "| files open: ", files_open)
    print(patch_batch)
print(track_files_open)
print(track_batch_size)

In [ ]:
files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
print(files_open)